In [1]:
from typing import TypedDict

from langgraph.graph import StateGraph,START,END
from langgraph.types import RetryPolicy
from loguru import  logger
from requests import HTTPError

class EmptyState(TypedDict):
    pass

def node_a(state:EmptyState) -> EmptyState:
    logger.info("node_a正在运行")
    raise HTTPError

builder=StateGraph(state_schema=EmptyState)
builder.add_node(
    "node_a",
    node_a,
    retry_policy=RetryPolicy(
        max_attempts=3,
        jitter=False
    )
)
builder.add_edge(START,"node_a")
builder.add_edge("node_a",END)
graph=builder.compile()

try:
    graph.invoke({})
except HTTPError as e:
    logger.info("重试次数耗尽:{}",e)

D:\miniconda\envs\langgraph\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
2026-09-22 17:20:09.465 | INFO     | __main__:node_a:12 - node_a正在运行
2026-09-22 17:20:09.966 | INFO     | __main__:node_a:12 - node_a正在运行
2026-09-22 17:20:10.967 | INFO     | __main__:node_a:12 - node_a正在运行
2026-09-22 17:20:10.969 | INFO     | __main__:<module>:31 - 重试次数耗尽:
